In [41]:
import torch
import pandas as pd
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [44]:
# загрузка датасета
raw = pd.read_csv('yelp_reviews.csv')

texts = raw['text']
labels = raw['label']

# разделение выборки на трейн и тест
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

train_texts = train_texts.reset_index(drop=True)
val_texts = val_texts.reset_index(drop=True)
train_labels = train_labels.reset_index(drop=True)
val_labels = val_labels.reset_index(drop=True)

# создание токенизатора с помощью класса AutoTokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# токенизируем тексты
train_texts_tokenized = tokenizer(list(train_texts), truncation=True)['input_ids']
val_texts_tokenized = tokenizer(list(val_texts), truncation=True)['input_ids']


# создаём класс кастомного, наследуясь от класса Dataset из PyTorch
class YelpDataset(Dataset):
    # в конструкторе просто сохраняем тексты и классы
    def __init__(self, texts, labels, max_len=256):
        self.texts = texts  # сохраните тексты
        self.labels = labels  # сохраните классы
        self.max_len = max_len  # сохрание max_len

    # возвращаем размер датасета (кол-во текстов)
    def __len__(self):
        return len(self.labels)  # верните размер датасета
        
    def __getitem__(self, idx):
        # возвращаем текст и его класс
        # для текста ограничиваем длину
        # не делаем никаких доп. преобразований как padding и masking
        return {
            'text': torch.tensor(self.texts[idx][:self.max_len]).long(),  # верните текст под индексом idx в виде тензора, ограничьте его длиной self.max_len
            'label': torch.tensor(self.labels[idx]).long(),  # верните класс под индексом idx в виде тензора
        }


# кастомная функция collate_fn для формирования батчей
def collate_fn(batch):
    texts = [torch.tensor(item['text']) for item in batch]
    labels = torch.tensor([item['label'] for item in batch])
    lengths = torch.tensor([len(seq) for seq in texts])
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return {
        'input_ids': padded_texts, 
        'lengths': lengths, 
        'labels': labels
    }


# создаём датасеты
train_dataset = YelpDataset(texts=train_texts_tokenized, labels=train_labels)
val_dataset = YelpDataset(texts=val_texts_tokenized, labels=val_labels)


batch_size = 64
# создаём даталоадеры
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f'Количество батчей в train_dataloader: {len(train_dataloader)}')
print(f'Количество батчей в val_dataloader: {len(val_dataloader)}')


print('Размерности батчей:')
for batch in train_dataloader:
    print('input_ids:', batch['input_ids'].shape)
    print('lengths:', batch['lengths'].shape)
    print('labels:', batch['labels'].shape)
    break 

Количество батчей в train_dataloader: 82
Количество батчей в val_dataloader: 21
Размерности батчей:
input_ids: torch.Size([64, 256])
lengths: torch.Size([64])
labels: torch.Size([64])


/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]


In [45]:
from torch.optim import Adam
from tqdm import tqdm
import torch.nn as nn
from sklearn.metrics import classification_report

In [59]:
# класс модели
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=embedding_dim, padding_idx=0)  # напишите слой эмбеддинга с входной размерной vocab_size и выходной embedding_dim
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)  # напишите слой RNN
        self.fc = nn.Linear(hidden_size, output_size)  # линейный слой для получения скоров классификации


    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)  # "запакуйте" тензор embedded, используя pack_padded_sequence
        packed_output, hidden = self.rnn(packed)  # посчитайте выход rnn
        output, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        
        # Используем последнее скрытое состояние для классификации
        out = self.fc(hidden[-1]) # посчитайте скоры для классификации по последнему скрытому состоянию (hidden[-1])
        return out


# создаём модель, оптимизатор, объявляем функцию потерь
vocab_size = tokenizer.vocab_size

model = SimpleRNN(vocab_size, embedding_dim=128, hidden_size=128, output_size=5)
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)


train_losses = []


n_epochs = 10


# реализуем обучение по эпохам
for epoch in range(n_epochs):
    model.train()
    total_train_loss, total_val_loss = 0., 0.
    for batch in tqdm(train_dataloader):
        inputs = batch["input_ids"]  # получите input_ids из батча
        lengths = batch["lengths"]  # получите длины текстов из батча 
        labels = batch["labels"]  # получите классы из батча

        optimizer.zero_grad()
        outputs = model(inputs, lengths)
        loss = loss_fn(outputs, labels)  # посчитайте функцию потерь

        # считаем градиенты 
        loss.backward()

        # обновляем веса
        optimizer.step()

        total_train_loss += loss.item()


    avg_train_loss = total_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)


    # подсчёт метрик по валидационному датасету
    y_true, y_pred = [], []
    for batch in tqdm(val_dataloader):
        inputs = batch['input_ids']
        lengths = batch['lengths']
        labels = batch['labels']


        with torch.no_grad():
            outputs = model(inputs, lengths)
            loss = loss_fn(outputs, labels)
            total_val_loss += loss.item()


            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())


        avg_val_loss = total_train_loss / len(train_dataloader)


    print(f"Epoch {epoch + 1}, Train Loss: {avg_train_loss:.4f}, Val loss: {avg_val_loss:.4f}")
    print('Метрики классификации')
    print(classification_report(y_true, y_pred))

  0%|          | 0/82 [00:00<?, ?it/s]/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]
100%|██████████| 21/21 [00:00<00:00, 21.30it/s]


Epoch 1, Train Loss: 1.6066, Val loss: 1.6066
Метрики классификации
              precision    recall  f1-score   support

           0       0.18      0.42      0.25       238
           1       0.25      0.11      0.16       301
           2       0.20      0.03      0.05       283
           3       0.20      0.36      0.25       242
           4       0.34      0.17      0.23       236

    accuracy                           0.21      1300
   macro avg       0.23      0.22      0.19      1300
weighted avg       0.23      0.21      0.18      1300



  0%|          | 0/82 [00:00<?, ?it/s]/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]
100%|██████████| 21/21 [00:00<00:00, 44.46it/s]


Epoch 2, Train Loss: 1.5557, Val loss: 1.5557
Метрики классификации
              precision    recall  f1-score   support

           0       0.30      0.10      0.15       238
           1       0.28      0.23      0.26       301
           2       0.24      0.59      0.34       283
           3       0.27      0.14      0.18       242
           4       0.34      0.22      0.27       236

    accuracy                           0.27      1300
   macro avg       0.29      0.26      0.24      1300
weighted avg       0.29      0.27      0.24      1300



  0%|          | 0/82 [00:00<?, ?it/s]/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]
100%|██████████| 21/21 [00:01<00:00, 16.14it/s]


...

Epoch 8, Train Loss: 0.9247, Val loss: 0.9247
Метрики классификации
              precision    recall  f1-score   support

           0       0.29      0.25      0.27       238
           1       0.28      0.27      0.28       301
           2       0.24      0.35      0.28       283
           3       0.23      0.19      0.21       242
           4       0.43      0.34      0.38       236

    accuracy                           0.28      1300
   macro avg       0.29      0.28      0.28      1300
weighted avg       0.29      0.28      0.28      1300



  0%|          | 0/82 [00:00<?, ?it/s]/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]
100%|██████████| 21/21 [00:00<00:00, 37.86it/s]


Epoch 9, Train Loss: 0.7461, Val loss: 0.7461
Метрики классификации
              precision    recall  f1-score   support

           0       0.31      0.26      0.29       238
           1       0.33      0.37      0.35       301
           2       0.25      0.21      0.23       283
           3       0.24      0.32      0.27       242
           4       0.34      0.29      0.31       236

    accuracy                           0.29      1300
   macro avg       0.29      0.29      0.29      1300
weighted avg       0.29      0.29      0.29      1300



  0%|          | 0/82 [00:00<?, ?it/s]/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_11247/1428157462.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  texts = [torch.tensor(item['text']) for item in batch]
100%|██████████| 21/21 [00:00<00:00, 37.72it/s]


Epoch 10, Train Loss: 0.6059, Val loss: 0.6059
Метрики классификации
              precision    recall  f1-score   support

           0       0.31      0.17      0.22       238
           1       0.34      0.45      0.38       301
           2       0.24      0.27      0.25       283
           3       0.25      0.26      0.26       242
           4       0.39      0.36      0.38       236

    accuracy                           0.31      1300
   macro avg       0.31      0.30      0.30      1300
weighted avg       0.31      0.31      0.30      1300

